In [0]:
# ===== 03_gold_aggregation =====

from pyspark.sql.functions import (
    col, sum as spark_sum, count as spark_count, count, when,
    date_format, round as spark_round, expr
)

silver_base = "/Volumes/dbacademy/default/ecommerce_project/silver"
gold_base = "/Volumes/dbacademy/default/ecommerce_project/gold"

# --- Load Silver tables ---
df_orders = spark.read.format("delta").load(f"{silver_base}/orders")
df_items = spark.read.format("delta").load(f"{silver_base}/order_items")
df_customers = spark.read.format("delta").load(f"{silver_base}/customers")
df_sellers = spark.read.format("delta").load(f"{silver_base}/sellers")
df_reviews = spark.read.format("delta").load(f"{silver_base}/order_reviews")
df_products = spark.read.format("delta").load(f"{silver_base}/products")
df_cat_translation = spark.read.format("delta").load(f"{silver_base}/category_translation")
df_mapping = spark.read.format("delta").load(f"{silver_base}/category_mapping")
df_rees46 = spark.read.format("delta").load(f"{silver_base}/rees46_events")

# --- 1. Combined Category Comparison (flagship multi-source table) ---
rees46_category_metrics = df_rees46.groupBy("category_level_1").agg(
    count(when(col("event_type") == "view", True)).alias("views"),
    count(when(col("event_type") == "cart", True)).alias("carts"),
    count(when(col("event_type") == "purchase", True)).alias("purchases")
).withColumnRenamed("category_level_1", "rees46_category_bucket")

olist_category_sales = df_items \
    .join(df_products, "product_id") \
    .join(df_cat_translation, "product_category_name") \
    .join(df_mapping, "product_category_name_english") \
    .join(df_orders.select("order_id", "is_delivered"), "order_id") \
    .filter(col("is_delivered") == True) \
    .groupBy("rees46_category_bucket") \
    .agg(
        spark_sum("price").alias("total_sales_revenue"),
        spark_count("order_id").alias("items_sold")
    )

combined_category_view = rees46_category_metrics.join(
    olist_category_sales, "rees46_category_bucket", "outer"
).fillna(0)

combined_category_view = combined_category_view.withColumn(
    "rees46_conversion_rate_pct",
    spark_round(expr("try_divide(purchases, views) * 100"), 2)
)
combined_category_view.write.format("delta").mode("overwrite").save(f"{gold_base}/category_comparison")
print("category_comparison:", combined_category_view.count())

# --- 2. Olist Monthly Sales Trend ---
monthly_sales = df_orders \
    .filter(col("is_delivered") == True) \
    .join(df_items, "order_id") \
    .withColumn("order_month", date_format(col("order_purchase_timestamp"), "yyyy-MM")) \
    .groupBy("order_month") \
    .agg(
        spark_sum("price").alias("total_revenue"),
        spark_count("order_id").alias("items_sold"),
        spark_count("order_item_id").alias("order_line_items")
    ) \
    .orderBy("order_month")
monthly_sales.write.format("delta").mode("overwrite").save(f"{gold_base}/olist_monthly_sales")
print("olist_monthly_sales:", monthly_sales.count())

# --- 3. Olist Customer Lifetime Value ---
customer_ltv = df_orders \
    .filter(col("is_delivered") == True) \
    .join(df_items, "order_id") \
    .join(df_customers, "customer_id") \
    .groupBy("customer_unique_id") \
    .agg(
        spark_sum("price").alias("lifetime_value"),
        spark_count("order_id").alias("total_orders")
    ) \
    .orderBy(col("lifetime_value").desc())
customer_ltv.write.format("delta").mode("overwrite").save(f"{gold_base}/olist_customer_ltv")
print("olist_customer_ltv:", customer_ltv.count())

# --- 4. Olist Seller Performance ---
seller_performance = df_orders \
    .filter(col("is_delivered") == True) \
    .join(df_items, "order_id") \
    .join(df_sellers, "seller_id") \
    .groupBy("seller_id", "seller_state") \
    .agg(
        spark_sum("price").alias("total_revenue"),
        spark_count("order_id").alias("items_sold")
    ) \
    .orderBy(col("total_revenue").desc())
seller_performance.write.format("delta").mode("overwrite").save(f"{gold_base}/olist_seller_performance")
print("olist_seller_performance:", seller_performance.count())

# --- 5. Olist Review Score Trends ---
review_trends = df_orders \
    .join(df_reviews, "order_id") \
    .withColumn("review_month", date_format(col("order_purchase_timestamp"), "yyyy-MM")) \
    .groupBy("review_month") \
    .agg(
        spark_sum("review_score").alias("total_score_sum"),
        spark_count("review_score").alias("review_count")
    ) \
    .withColumn("avg_review_score", spark_round(col("total_score_sum") / col("review_count"), 2)) \
    .orderBy("review_month")
review_trends.write.format("delta").mode("overwrite").save(f"{gold_base}/olist_review_trends")
print("olist_review_trends:", review_trends.count())

# --- 6. REES46 Overall Funnel ---
overall_funnel = df_rees46.agg(
    count(when(col("event_type") == "view", True)).alias("total_views"),
    count(when(col("event_type") == "cart", True)).alias("total_carts"),
    count(when(col("event_type") == "purchase", True)).alias("total_purchases")
)
overall_funnel = overall_funnel \
    .withColumn("view_to_cart_rate_pct", spark_round(expr("try_divide(total_carts, total_views) * 100"), 2)) \
    .withColumn("cart_to_purchase_rate_pct", spark_round(expr("try_divide(total_purchases, total_carts) * 100"), 2)) \
    .withColumn("overall_conversion_rate_pct", spark_round(expr("try_divide(total_purchases, total_views) * 100"), 2))
overall_funnel.write.format("delta").mode("overwrite").save(f"{gold_base}/rees46_overall_funnel")
print("rees46_overall_funnel written")

print("=== Gold aggregation complete ===")

category_comparison: 15
olist_monthly_sales: 23
olist_customer_ltv: 93358
olist_seller_performance: 2970
olist_review_trends: 25
rees46_overall_funnel written
=== Gold aggregation complete ===
